In [ ]:

import os
import sys

REPO_URL = "https://github.com/Baoshan-Song/KFV-FGO-Comparison.git"
BRANCH = "python_colab"
PROJECT_DIR = "/content/KFV-FGO-Comparison"


if os.path.exists(PROJECT_DIR):
  !rm -rf {PROJECT_DIR}


print(f"🚀 Cloning branch '{BRANCH}' from GitHub...")
!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}


if os.path.exists(PROJECT_DIR):
  os.chdir(PROJECT_DIR)
  if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
  print(f"\n✅ Successfully directory switched to:\n   {os.getcwd()}")
else:
  print("❌ Error: Repository cloning failed!")


PROJECT_PATH = "/content/KFV-FGO-Comparison"

Google Drive mount is not required; using the cloned project directory.
Drive exists: True
Drive writable: True
✅ Active Working Directory set to:
   /content/KFV-FGO-Comparison



In [ ]:
# @title Example 1: Interactive Synthetic Data Generator
# @markdown Drag the sliders in the right panel to configure the anchor distribution radius and GMM noise parameters. Running this cell will invoke `example_simulation_data.py` with CLI arguments to generate the corresponding `.mat` dataset.

import os
import sys

# ==============================================================================
# 0. Set Working Directory & Import Python Paths
# ==============================================================================
PROJECT_PATH = "/content/KFV-FGO-Comparison"

if os.path.exists(PROJECT_PATH):
  os.chdir(PROJECT_PATH)
  if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
  print(f"✅ Active Working Directory set to:\n   {os.getcwd()}\n")
else:
  print(f"❌ Error: Directory '{PROJECT_PATH}' not found!")
  print("   Please check if Google Drive is mounted properly.\n")

# ==============================================================================
# 🎛️ Interactive Form Controls (Units safely embedded in Markdown Labels)
# ==============================================================================

# @markdown ### 📍 1. Anchor Array Geometry Setup
# @markdown **Anchor Distribution Radius [m]:**
anchor_radius = 200  # @param {type:"slider", min:10, max:500, step:10}

# @markdown ### 📉 2. GMM Noise Parameters (Non-Gaussian / NLOS Noise Configuration)
# @markdown **Component 1 (LOS Line-of-Sight Noise):**
# @markdown - Weight w1 (relative proportion):
gmm_w1 = 0.8  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

# @markdown - Mean mu1 [m]:
gmm_mu1 = 0.0  # @param {type:"slider", min:-5.0, max:5.0, step:0.5}

# @markdown - Standard Deviation sigma1 [m]:
gmm_sigma1 = 0.1  # @param {type:"slider", min:0.01, max:2.0, step:0.05}

# @markdown **Component 2 (NLOS Non-Line-of-Sight / Outliers):**
# @markdown - Weight w2 (relative proportion):
gmm_w2 = 0.2  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

# @markdown - Mean mu2 [m]:
gmm_mu2 = 30.0  # @param {type:"slider", min:0.0, max:100.0, step:5.0}

# @markdown - Standard Deviation sigma2 [m]:
gmm_sigma2 = 5.0  # @param {type:"slider", min:0.5, max:20.0, step:0.5}

# @markdown ### 💾 3. Export Settings
# @markdown **Output Dataset Filename (.mat):**
output_mat_name = "circle_cv_gmm_L4.mat"  # @param {type:"string"}

# ==============================================================================
# 🚀 4. Run Pure Data Generation with CLI Arguments
# ==============================================================================

# Normalize GMM weights (ensure w1 + w2 = 1.0)
total_w = gmm_w1 + gmm_w2
w1_norm = gmm_w1 / total_w if total_w > 0 else 1.0
w2_norm = gmm_w2 / total_w if total_w > 0 else 0.0

print(
    f"🚀 Executing 'example_simulation_data.py' with CLI physical"
    " arguments...\n"
)
print(f"   • Anchor Array Radius : {anchor_radius} m")
print(
    f"   • LOS Noise (Comp 1)  : w = {w1_norm:.2f}, μ = {gmm_mu1:.1f} m, σ ="
    f" {gmm_sigma1:.2f} m"
)
print(
    f"   • NLOS Noise (Comp 2) : w = {w2_norm:.2f}, μ = {gmm_mu2:.1f} m, σ ="
    f" {gmm_sigma2:.2f} m"
)
print(f"   • Output Filename     : {output_mat_name}")
print("-" * 75 + "\n")

# Format and execute the shell command with interactive parameter flags
!python example_simulation_data.py \
    --anchor_radius {anchor_radius} \
    --gmm_w1 {w1_norm} \
    --gmm_w2 {w2_norm} \
    --gmm_mu1 {gmm_mu1} \
    --gmm_mu2 {gmm_mu2} \
    --gmm_sigma1 {gmm_sigma1} \
    --gmm_sigma2 {gmm_sigma2} \
    --output_filename {output_mat_name}

✅ Active Working Directory set to:
   /content/KFV-FGO-Comparison

🚀 Executing 'example_simulation_data.py' with CLI physical arguments...

   • Anchor Array Radius : 200 m
   • LOS Noise (Comp 1)  : w = 0.80, μ = 0.0 m, σ = 0.10 m
   • NLOS Noise (Comp 2) : w = 0.20, μ = 30.0 m, σ = 5.00 m
   • Output Filename     : circle_cv_gmm_L4.mat
---------------------------------------------------------------------------

✅ Simulation Data Generated & Saved Successfully!
   • File Path        : /content/KFV-FGO-Comparison/data/circle_cv_gmm_L4.mat
   • Anchor Radius    : 200.0 m
   • GMM Weights (w)  : (0.80, 0.20)
   • GMM Means (μ)    : (0.0, 30.0) m
   • GMM Sigmas (σ)   : (0.1, 5.0) m


In [ ]:
# @title Example 2.1: KFV vs. Re-FGO (Constant Velocity & UWB Range Localization)
# @markdown Adjust the sliders and dropdowns below to configure KFV and FGO parameters for 2D Constant Velocity (CV) motion and Ultra-Wideband (UWB) range positioning. Running this cell automatically writes parameters to `config/kfv_fgo_comparison_test.json` and executes `example_kfv_fgo_comparison.py`.

import json
import os
import sys

# ===============================================================================
# 0. Working Directory Setup & Path Injection
# ===============================================================================
PROJECT_PATH = "/content/KFV-FGO-Comparison"

if os.path.exists(PROJECT_PATH):
  os.chdir(PROJECT_PATH)
  if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
  print(f"✅ Active Working Directory set to:\n   {os.getcwd()}\n")
else:
  print(f"❌ Error: Directory '{PROJECT_PATH}' not found!")
  print("   Please check if the repository was cloned successfully.\n")

# ===============================================================================
# 🎛️ Interactive Form Controls (Units embedded cleanly into Label Text)
# ===============================================================================

# @markdown ### 📍 1. Initial State & Dataset Setup
data_filename = "circle_cv_gmm_L4.mat"  # @param {type:"string"}

# @markdown **Initial Position Error X [m]:**
err_x = 100.0  # @param {type:"slider", min:-200, max:200, step:10}

# @markdown **Initial Position Error Y [m]:**
err_y = -100.0  # @param {type:"slider", min:-200, max:200, step:10}

# @markdown ### ⚙️ 2. KFV Estimator Options
kfv_mode = "EKF"  # @param ["EKF", "IEKF", "UKF"]
max_iteration = 2  # @param {type:"slider", min:1, max:20, step:1}

# @markdown **Convergence Threshold [m]:**
thres_iteration = 1e-6  # @param {type:"number"}

robust_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]

# @markdown **Huber Tuning Parameter delta [m]:**
robust_delta = 2.0  # @param {type:"slider", min:0.5, max:10.0, step:0.5}

# @markdown **Sliding Window Size N [steps]:**
window_size = 1  # @param {type:"slider", min:1, max:20, step:1}

# @markdown ### 🔄 3. FGO Equivalence & AutoDiff Options
imitate_KFV = True  # @param {type:"boolean"}
autoDiff = True  # @param {type:"boolean"}

# ===============================================================================
# 2. JSON Configuration Template & Dynamic Update (Clean Numeric Data)
# ===============================================================================

config_data = {
    "data": {"mode": "sim", "path": data_filename},
    "KFV": {
        "mode": kfv_mode,
        "dt": 1.0,
        "errX0": [float(err_x), float(err_y), 0.0, 0.0],
        "P0": [
            [50.0, 0.0, 0.0, 0.0],
            [0.0, 50.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ],
        "Q": [
            [0.0001, 0.0, 0.0, 0.0],
            [0.0, 0.0001, 0.0, 0.0],
            [0.0, 0.0, 0.0001, 0.0],
            [0.0, 0.0, 0.0, 0.0001],
        ],
        "omega": 0.06283185307179587,
        "R": 0.01,
        "max_iteration": int(max_iteration),
        "thres_iteration": float(thres_iteration),
        "robust_kernel": robust_kernel,
        "robust_delta": float(robust_delta),
        "window_size": int(window_size),
    },
    "FGO": {"imitate_KFV": imitate_KFV, "autoDiff": autoDiff},
}

json_filename = "config/kfv_fgo_comparison_test.json"
os.makedirs(os.path.dirname(json_filename), exist_ok=True)

# Write pure numeric dictionary to JSON file
with open(json_filename, "w", encoding="utf-8") as f:
  json.dump(config_data, f, indent=2)

# ===============================================================================
# 3. Terminal Print Output (Displaying Explicit Physical Units)
# ===============================================================================

print(f"✅ Successfully updated '{json_filename}' with physical parameters:")
print(f"   • Dataset Path        : {data_filename}")
print(f"   • KFV Variant         : {kfv_mode}")
print(f"   • Sampling Time dt    : 1.0 s")
print(f"   • Initial Err Vector  : [{err_x:.1f} m, {err_y:.1f} m, 0.0 m/s, 0.0 rad/s]")
print(f"   • State Covariance P0 : diag([50.0 m², 50.0 m², 1.0 (m/s)², 1.0 (rad/s)²])")
print(f"   • Process Noise Q     : diag([1e-4 m², 1e-4 m², 1e-4 (m/s)², 1e-4 (rad/s)²])")
print(f"   • Angular Vel ω       : 0.0628 rad/s (Period T ≈ 100.0 s)")
print(f"   • UWB Noise Var R     : 0.0100 m² (Standard Deviation σ = 0.10 m)")
print(f"   • Robust Loss Kernel  : {robust_kernel} (Threshold δ = {robust_delta:.1f} m)")
print(f"   • Convergence Thres   : {thres_iteration:.1e} m (Max Iterations = {max_iteration})")
print(f"   • Sliding Window N    : {window_size} step(s)")
print(f"   • Equivalence Flags   : imitate_KFV = {imitate_KFV} | AutoDiff = {autoDiff}")
print("-" * 75)

# ===============================================================================
# 4. Execute Script (Invoke example_kfv_fgo_comparison.py)
# ===============================================================================

script_name = "example_kfv_fgo_comparison.py"

if os.path.exists(script_name):
  print(f"🚀 Running '{script_name}' for UWB/CV localization benchmark...\n")
  !python example_kfv_fgo_comparison.py
else:
  print(
      f"⚠️ Warning: Could not find '{script_name}' in current working"
      f" directory ({os.getcwd()})."
  )
  print("Please make sure your repository code is added to the Python path.")

✅ Active Working Directory set to:
   /content/KFV-FGO-Comparison

✅ Successfully updated 'config/kfv_fgo_comparison_test.json' with physical parameters:
   • Dataset Path        : circle_cv_gmm_L4.mat
   • KFV Variant         : EKF
   • Sampling Time dt    : 1.0 s
   • Initial Err Vector  : [100.0 m, -100.0 m, 0.0 m/s, 0.0 rad/s]
   • State Covariance P0 : diag([50.0 m², 50.0 m², 1.0 (m/s)², 1.0 (rad/s)²])
   • Process Noise Q     : diag([1e-4 m², 1e-4 m², 1e-4 (m/s)², 1e-4 (rad/s)²])
   • Angular Vel ω       : 0.0628 rad/s (Period T ≈ 100.0 s)
   • UWB Noise Var R     : 0.0100 m² (Standard Deviation σ = 0.10 m)
   • Robust Loss Kernel  : huber (Threshold δ = 2.0 m)
   • Convergence Thres   : 1.0e-06 m (Max Iterations = 2)
   • Sliding Window N    : 1 step(s)
   • Equivalence Flags   : imitate_KFV = True | AutoDiff = True
---------------------------------------------------------------------------
🚀 Running 'example_kfv_fgo_comparison.py' for UWB/CV localization benchmark...

KFV Estim

In [ ]:
# @title Example 2.2: SW-FGO Configuration (Sliding Window Factor Graph Optimization)
# @markdown Adjust the sliders or select parameters in the right panel, then click run. This cell automatically writes parameters to `config/swfgo_test.json` and executes `example_sw_fgo.py`.

import json
import os
import sys

# ===============================================================================
# 0. Working Directory Setup & Path Injection
# ===============================================================================
PROJECT_PATH = "/content/KFV-FGO-Comparison"

if os.path.exists(PROJECT_PATH):
  os.chdir(PROJECT_PATH)
  if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
  print(f"✅ Active Working Directory set to:\n   {os.getcwd()}\n")
else:
  print(f"❌ Error: Directory '{PROJECT_PATH}' not found!")
  print("   Please check if the repository was cloned successfully.\n")

# ===============================================================================
# 🎛️ Interactive Form Controls (Units safely embedded in Markdown Labels)
# ===============================================================================

# @markdown ### 📍 1. Initial State Setup
# @markdown **Initial Position Error X [m]:**
err_x = 0.0  # @param {type:"slider", min:-200, max:200, step:10}

# @markdown **Initial Position Error Y [m]:**
err_y = 0.0  # @param {type:"slider", min:-200, max:200, step:10}

# @markdown ### ⚙️ 2. Estimator & Robust Loss Options
robust_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]

# @markdown **Huber Loss Tuning Threshold delta [m]:**
robust_delta = 2.0  # @param {type:"slider", min:0.5, max:10.0, step:0.5}

# @markdown **Sliding Window Size N [steps]:**
window_size = 1  # @param {type:"slider", min:1, max:20, step:1}

# @markdown ### 🔄 3. Mode & Algorithmic Equivalence Flags
imitate_KFV = False  # @param {type:"boolean"}
autoDiff = False  # @param {type:"boolean"}

# ===============================================================================
# 2. JSON Configuration Template & Dynamic Update (Clean Numeric Types)
# ===============================================================================

config_data = {
    "data": {"mode": "sim", "path": "circle_cv_gmm_L4.mat"},
    "FGO": {
        "dt": 1.0,
        "errX0": [float(err_x), float(err_y), 0.0, 0.0],
        "P0": [
            [50.0, 0.0, 0.0, 0.0],
            [0.0, 50.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ],
        "Q": [
            [0.0001, 0.0, 0.0, 0.0],
            [0.0, 0.0001, 0.0, 0.0],
            [0.0, 0.0, 0.0001, 0.0],
            [0.0, 0.0, 0.0, 0.0001],
        ],
        "omega": 0.06283185307179587,
        "R": 0.01,
        "max_iteration": 1,
        "thres_iteration": 1e-6,
        "robust_kernel": robust_kernel,
        "robust_delta": float(robust_delta),
        "window_size": int(window_size),
        "imitate_KFV": imitate_KFV,
        "autoDiff": autoDiff,
    },
}

json_filename = "config/swfgo_test.json"
os.makedirs(os.path.dirname(json_filename), exist_ok=True)

# Write clean numeric dictionary to JSON file
with open(json_filename, "w", encoding="utf-8") as f:
  json.dump(config_data, f, indent=2)

# ===============================================================================
# 3. Terminal Print Output (Displaying Explicit Physical Units)
# ===============================================================================

print(f"✅ Successfully updated '{json_filename}' with physical parameters:")
print(
    f"   • Initial Err Vector  : [{err_x:.1f} m, {err_y:.1f} m, 0.0 m/s, 0.0"
    " rad/s]"
)
print(
    "   • State Covariance P0 : diag([50.0 m², 50.0 m², 1.0 (m/s)², 1.0"
    " (rad/s)²])"
)
print(
    "   • Process Noise Q     : diag([1e-4 m², 1e-4 m², 1e-4 (m/s)², 1e-4"
    " (rad/s)²])"
)
print("   • Sampling Time dt    : 1.0 s")
print("   • Angular Vel ω       : 0.0628 rad/s (Period T ≈ 100.0 s)")
print("   • UWB Noise Var R     : 0.0100 m² (Standard Deviation σ = 0.10 m)")
print(
    f"   • Robust Loss Kernel  : {robust_kernel} (Threshold δ ="
    f" {robust_delta:.1f} m)"
)
print(f"   • Sliding Window N    : {window_size} step(s)")
print(
    f"   • Equivalence Flags   : imitate_KFV = {imitate_KFV} | AutoDiff ="
    f" {autoDiff}"
)
print("-" * 75)

# ===============================================================================
# 4. Execute Script (Invoke example_sw_fgo.py)
# ===============================================================================

script_name = "example_sw_fgo.py"

if os.path.exists(script_name):
  print(f"🚀 Running '{script_name}' for SW-FGO simulation...\n")
  !python example_sw_fgo.py
else:
  print(
      f"⚠️ Warning: Could not find '{script_name}' in current working"
      f" directory ({os.getcwd()})."
  )
  print("Please make sure your repository code is added to the Python path.")

✅ Active Working Directory set to:
   /content/KFV-FGO-Comparison

✅ Successfully updated 'config/swfgo_test.json' with physical parameters:
   • Initial Err Vector  : [0.0 m, 0.0 m, 0.0 m/s, 0.0 rad/s]
   • State Covariance P0 : diag([50.0 m², 50.0 m², 1.0 (m/s)², 1.0 (rad/s)²])
   • Process Noise Q     : diag([1e-4 m², 1e-4 m², 1e-4 (m/s)², 1e-4 (rad/s)²])
   • Sampling Time dt    : 1.0 s
   • Angular Vel ω       : 0.0628 rad/s (Period T ≈ 100.0 s)
   • UWB Noise Var R     : 0.0100 m² (Standard Deviation σ = 0.10 m)
   • Robust Loss Kernel  : none (Threshold δ = 2.0 m)
   • Sliding Window N    : 1 step(s)
   • Equivalence Flags   : imitate_KFV = False | AutoDiff = False
---------------------------------------------------------------------------
🚀 Running 'example_sw_fgo.py' for SW-FGO simulation...

FGO Estimator Statistics:
  MSE:                    0.001048
  RMSE:                   0.032373
  MAE:                    0.029408
  Max Error:              0.079126
  95% Absolute Error: